# Debugging Patterns in Python

Debugging is the process of finding out why code behaves differently from what we expected. Good debugging is systematic: reproduce the problem, inspect the state, isolate the cause, and verify the fix.

In this notebook, we will cover:
1. Reading a traceback to locate a failure.
2. Adding targeted logging to inspect program state.
3. Using `breakpoint()` and `pdb` commands.
4. Following a repeatable debugging workflow.

## 1. Reading a Traceback

Python's traceback shows the path that led to an error. Read it from the bottom up: the final line gives the exception, and the lines above show the calls and line numbers that produced it.

In [1]:
import traceback


def average_feature_value(rows):
    total = sum(rows)
    return total / len(rows)


try:
    average_feature_value([])
except ZeroDivisionError:
    traceback.print_exc()

Traceback (most recent call last):
  File "C:\Users\Acer\AppData\Local\Temp\ipykernel_26320\731980486.py", line 10, in <module>
    average_feature_value([])
    ~~~~~~~~~~~~~~~~~~~~~^^^^
  File "C:\Users\Acer\AppData\Local\Temp\ipykernel_26320\731980486.py", line 6, in average_feature_value
    return total / len(rows)
           ~~~~~~^~~~~~~~~~~
ZeroDivisionError: division by zero


The final line identifies the exception (`ZeroDivisionError`). The frames above show the call chain, including the function and line where the division happened. The goal is not to read every line, but to find the first frame in your own code that explains the state.

## 2. Targeted Logging

Print statements are quick, but `logging` lets us control severity and turn diagnostics on or off. Log the inputs and intermediate values at the point where the behavior starts to diverge from expectations.

In [2]:
import logging


logger = logging.getLogger("debugging_demo")
logger.setLevel(logging.DEBUG)

if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))
    logger.addHandler(handler)


def average_feature_value(rows):
    logger.debug("Received %d row(s)", len(rows))
    if not rows:
        raise ValueError("rows cannot be empty")

    total = sum(rows)
    logger.debug("Feature total: %s", total)
    return total / len(rows)


try:
    average_feature_value([])
except ValueError as error:
    logger.error("Could not calculate average: %s", error)

print("Average:", average_feature_value([1.2, 3.4, 2.1]))

DEBUG: Received 0 row(s)


ERROR: Could not calculate average: rows cannot be empty


DEBUG: Received 3 row(s)


DEBUG: Feature total: 6.7


Average: 2.2333333333333334


## 3. Using `breakpoint()` and `pdb`

The built-in `breakpoint()` pauses execution and opens the Python debugger. From there, you can inspect variables and step through code. This only works in an interactive terminal session, so the call below is left commented out for the notebook.

In [3]:
def inspect_training_batch(batch):
    total = sum(batch)
    # breakpoint()  # Run this function in a terminal, then inspect values with pdb.
    return total / len(batch)


print("Batch average:", inspect_training_batch([10, 20, 30]))

Batch average: 20.0


Common `pdb` commands:
* `p variable` prints a value.
* `n` executes the next line without entering called functions.
* `s` steps into a function call.
* `c` continues until the next breakpoint or the program ends.
* `q` quits the debugger.

## 4. A Repeatable Debugging Workflow

1. **Reproduce** the failure with the smallest input you can.
2. **Read** the traceback and identify the failing line.
3. **Inspect** the state immediately before that line.
4. **Isolate** one assumption at a time instead of changing several things at once.
5. **Verify** the fix with a test so the same bug does not return.

In [4]:
def safe_divide(numerator, denominator):
    if denominator == 0:
        raise ValueError("denominator cannot be zero")
    return numerator / denominator


try:
    safe_divide(10, 0)
except ValueError as error:
    print("Expected error:", error)

print("Valid result:", safe_divide(10, 2))

Expected error: denominator cannot be zero
Valid result: 5.0
